# Lab 05: Movement Invariants & Trajectory Planning

**Computational Sensorimotor Control — Week 5**

**Objectives:**
- Reproduce Morasso's observation: straight hand paths, complex joint trajectories
- Implement the minimum-jerk trajectory (Flash & Hogan, 1985)
- Compare EPH vs minimum jerk on center-out reaching
- Implement the VITE model (Bullock & Grossberg, 1988) using `rk4_step`
- Demonstrate VITE speed scaling via the Go signal
- Run a three-way comparison: EPH vs min-jerk vs VITE
- Compare via-point handling across all three controllers

**Key equations:**

Minimum jerk: $x(\tau) = x_0 + (x_f - x_0) \cdot (10\tau^3 - 15\tau^4 + 6\tau^5), \quad \tau = t/T$

VITE: $\dot{V} = \alpha[-V + G(t) \cdot (T - P)], \quad \dot{P} = V$

---
## Part 1: Setup

Install and import the `smc` plant library, exactly as in Lab 04.

In [ ]:
# Install the smc library (only needs to run once per session)
!pip3 install git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

from smc import (
    Arm, make_muscles, simulate_lambda,
    Q_REF, L1, L2, rk4_step,
)

arm = Arm()
ik = arm.inverse_kinematics
start_hand = arm.forward_kinematics(Q_REF)
print(f"Reference posture: θ₁={np.rad2deg(Q_REF[0]):.0f}°, θ₂={np.rad2deg(Q_REF[1]):.0f}°")
print(f"Hand position: ({start_hand[0]*100:.1f}, {start_hand[1]*100:.1f}) cm")

# ── Movement parameters (shared across all parts) ──────────────────────
REACH_RADIUS = 0.08     # 8 cm center-out amplitude
MJ_DURATION  = 0.500    # 500 ms for minimum jerk
C_CMD        = 0.25     # C-command for EPH (rad)
DT_VITE      = 0.0002   # 0.2 ms integration step for VITE
COLORS_8 = ['#E74C3C','#E67E22','#27AE60','#2E86AB','#3498db','#8e44ad','#E8735A','#7f8c8d']

# ── Lab 04 helper functions (provided — same code from last week) ──────
def lambda_for_posture(q_target, C=C_CMD):
    muscles = make_muscles()
    return np.array([m.length(q_target) - (abs(m.r_sh) + abs(m.r_el)) * C
                     for m in muscles])

def make_ramp(lam_init, lam_final, t_start=0.05, duration=0.30):
    def lam_fn(t):
        if t < t_start:
            return lam_init.copy()
        elif t < t_start + duration:
            return lam_init + (t - t_start) / duration * (lam_final - lam_init)
        else:
            return lam_final.copy()
    return lam_fn

# ── Pre-compute 8 center-out targets ───────────────────────────────────
targets_8 = []
angles_8 = np.arange(0, 360, 45)
for deg in angles_8:
    tx = start_hand[0] + REACH_RADIUS * np.cos(np.radians(deg))
    ty = start_hand[1] + REACH_RADIUS * np.sin(np.radians(deg))
    targets_8.append(np.array([tx, ty]))
print(f"\n8 targets at {REACH_RADIUS*100:.0f} cm from Q_REF:")
for deg, tgt in zip(angles_8, targets_8):
    print(f"  {deg:3d}°: ({tgt[0]*100:.1f}, {tgt[1]*100:.1f}) cm")

---
## Part 2: Morasso's Observation (Lecture §1 — Figure 1)

In 1981, Pietro Morasso found that hand paths are approximately straight and velocity profiles bell-shaped, but **joint-angle trajectories are complex and direction-dependent.** The nervous system appears to plan in task coordinates.

We first need the minimum-jerk function (Flash & Hogan, 1985) as a ground-truth trajectory generator. Then we reproduce Morasso's 3-panel figure.

In [ ]:
def minimum_jerk(p0, pf, T, dt=0.001):
    """Compute minimum-jerk trajectory from p0 to pf in time T.
    Returns: t, pos, vel
    """
    t = np.arange(0, T + dt, dt)
    tau = t / T
    s  = 10*tau**3 - 15*tau**4 + 6*tau**5          # position basis
    sd = (30*tau**2 - 60*tau**3 + 30*tau**4) / T    # velocity basis
    d = (pf - p0)[None, :]
    pos = p0[None, :] + s[:, None] * d
    vel = sd[:, None] * d
    return t, pos, vel

# ── Reproduce Figure 1: Morasso center-out ──
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Panel A: Hand paths
ax = axes[0]
ax.plot(start_hand[0]*100, start_hand[1]*100, 'ko', ms=8, zorder=5)
for i, tgt in enumerate(targets_8):
    t, pos, vel = minimum_jerk(start_hand, tgt, MJ_DURATION)
    ax.plot(pos[:, 0]*100, pos[:, 1]*100, color=COLORS_8[i], lw=1.8)
    ax.plot(tgt[0]*100, tgt[1]*100, 'o', color=COLORS_8[i], ms=5)
ax.set_aspect('equal'); ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.set_title('A. Hand Paths')

# Panel B: Velocity profiles
ax = axes[1]
for i, tgt in enumerate(targets_8):
    t, pos, vel = minimum_jerk(start_hand, tgt, MJ_DURATION)
    ax.plot(t*1000, np.linalg.norm(vel, axis=1)*100, color=COLORS_8[i], lw=1.5)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Speed (cm/s)')
ax.set_title('B. Velocity Profiles')

# Panel C: Joint angles (IK at each waypoint)
ax = axes[2]
for i, tgt in enumerate(targets_8):
    t, pos, vel = minimum_jerk(start_hand, tgt, MJ_DURATION)
    q1s = [np.degrees(ik(p[0], p[1])[0]) for p in pos]
    q2s = [np.degrees(ik(p[0], p[1])[1]) for p in pos]
    ax.plot(t*1000, q1s, color=COLORS_8[i], lw=1.0, ls='-')
    ax.plot(t*1000, q2s, color=COLORS_8[i], lw=1.0, ls='--')
ax.plot([], [], 'k-', label='Shoulder'); ax.plot([], [], 'k--', label='Elbow')
ax.legend(fontsize=9); ax.set_xlabel('Time (ms)'); ax.set_ylabel('Angle (°)')
ax.set_title('C. Joint Angles')

fig.suptitle('Figure 1: Morasso Center-Out (min-jerk, 8 cm, T = 500 ms)', fontweight='bold', y=1.02)
plt.tight_layout()
print("Straight hand paths, bell-shaped velocity, but COMPLEX joint trajectories.")
print("The joint angles in C are the IK solution at every waypoint of the hand paths in A.")

---
## Part 3: Minimum-Jerk Profiles (Lecture §2 — Figure 2)

The minimum-jerk model minimizes $\int_0^T \|\dddot{x}\|^2 dt$. The result is a 5th-order polynomial with a perfectly symmetric bell-shaped velocity, zero-crossing acceleration at midpoint, and minimized jerk.

**Key point from lecture:** Min-jerk is **open-loop by nature** — a reference trajectory computed once before movement. No feedback, no correction. The purest feedforward plan.

In [ ]:
# Normalized profiles (T=1, amplitude=1)
t_n, pos_n, vel_n = minimum_jerk(np.array([0., 0.]), np.array([1., 0.]), 1.0)
# Also compute acceleration and jerk
tau = t_n
sdd  = (60*tau - 180*tau**2 + 120*tau**3)    # accel basis (T=1)
sddd = (60 - 360*tau + 360*tau**2)             # jerk basis (T=1)

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
TEAL = '#2E86AB'
for ax, (y, title, ylab) in zip(axes, [
    (pos_n[:,0], 'Position', 'x(τ)'),
    (vel_n[:,0], 'Velocity', 'ẋ(τ)'),
    (sdd, 'Acceleration', 'ẍ(τ)'),
    (sddd, 'Jerk', 'x⃛(τ)')]):
    ax.fill_between(t_n, 0, y, alpha=0.25, color=TEAL)
    ax.plot(t_n, y, color=TEAL, lw=2)
    ax.axhline(0, color='#CCC', lw=0.5, zorder=0)
    ax.set_xlabel('τ = t/T'); ax.set_ylabel(ylab); ax.set_title(title)

fig.suptitle('Figure 2: Minimum-Jerk Profiles (shading for visual clarity — does not represent integral area)',
             fontsize=10, y=-0.01)
plt.tight_layout()

---
## Part 4: EPH vs Minimum Jerk — Head to Head (Lecture §3 — Figure 3)

Now the critical comparison. Run both controllers on the same 8 center-out targets.

**Your tasks:**
1. Create a figure with 2 side-by-side panels (EPH left, Min-Jerk right)
2. For each of the 8 targets:
   - **EPH:** compute `target_q` via `ik()`, get λ ramps, call `simulate_lambda()`
   - **Min-jerk:** call `minimum_jerk()` (already done in Part 2, same code)
3. Plot hand paths in both panels with the same axis limits
4. EPH paths should **curve**; min-jerk paths should be **straight**

In [ ]:
# TODO: COMPLETE THIS CELL
# Reproduce Figure 3 from the lecture notes.
# Hint: EPH code is the same as Lab 04 Part 6.




---
## Part 5: Implementing the VITE Model (Lecture §4 — Figure 4)

Bullock & Grossberg (1988) proposed a neural circuit that produces min-jerk-like trajectories **without** solving an optimization problem:

$$\dot{V} = \alpha[-V + G(t) \cdot (T - P)], \quad \dot{P} = V$$

State vector: $\mathbf{x} = [P_x, P_y, V_x, V_y]^T$.  Critical damping: $\alpha = 4G$.

**DV = T − P** is the difference vector — note that it is **not** an independent state variable. It is derived from the planned position P (a true state) and the fixed target T. Its exponential decay is a consequence of the V̇ dynamics.

**Your tasks:**
1. Complete `vite_deriv()` — the derivative function for `rk4_step`
2. Complete `sim_vite()` — fill in the two TODO lines in the simulation loop
3. Run the test cell to verify your implementation reaches the target

In [ ]:
def vite_deriv(state, target, G, alpha):
    """VITE derivative: returns d/dt [Px, Py, Vx, Vy].
    
    Ṗx = Vx,  Ṗy = Vy
    V̇x = alpha * (-Vx + G * (Tx - Px))
    V̇y = alpha * (-Vy + G * (Ty - Py))
    """
    Px, Py, Vx, Vy = state
    # TODO: return the 4-element derivative array
    pass

def sim_vite(p0, target, G_amp, T=1.2, dt=DT_VITE):
    """Simulate VITE with critical damping α = 4G.
    
    Returns: ts, pos, vel, dv_mag
        ts     — time array
        pos    — (n, 2) planned positions
        vel    — (n, 2) planned velocities  
        dv_mag — (n,) |T - P|, the difference vector magnitude
    """
    alpha = 4.0 * G_amp
    state = np.array([p0[0], p0[1], 0.0, 0.0])
    n = int(T / dt)
    ts = np.zeros(n); pos = np.zeros((n, 2)); vel = np.zeros((n, 2))
    dv_mag = np.zeros(n)
    for i in range(n):
        ts[i] = i * dt
        pos[i] = state[:2]
        vel[i] = state[2:]
        # TODO: (1) compute dv_mag[i] = distance from current position to target
        # TODO: (2) advance state using rk4_step with vite_deriv
        pass
    return ts, pos, vel, dv_mag

# ── Test your implementation ──
tv, pv, vv, dv = sim_vite(start_hand, targets_8[0], G_amp=10, T=1.0)
err = np.linalg.norm(pv[-1] - targets_8[0]) * 100
print(f"VITE final pos:  ({pv[-1,0]*100:.1f}, {pv[-1,1]*100:.1f}) cm")
print(f"Target:          ({targets_8[0][0]*100:.1f}, {targets_8[0][1]*100:.1f}) cm")
print(f"Endpoint error:  {err:.3f} cm {'✓' if err < 0.1 else '✗ — check your code'}")

---
### Part 5b: VITE vs Min-Jerk — Single Reach (Lecture §4 — Figure 4)

Now compare VITE at two Go signal values (G = 3 slow, G = 10 fast) against minimum jerk (T = 500 ms). The min-jerk peak should fall **between** the two VITE speeds in both peak velocity and movement duration.

In [ ]:
tgt = targets_8[0]  # rightward

# Min-jerk
t_m, pos_m, vel_m = minimum_jerk(start_hand, tgt, MJ_DURATION)
spd_m = np.linalg.norm(vel_m, axis=1)

# VITE slow (G=3, α=12) and fast (G=10, α=40)
tv_s, pv_s, vv_s, dv_s = sim_vite(start_hand, tgt, G_amp=3, T=1.5)
sv_s = np.linalg.norm(vv_s, axis=1)
tv_f, pv_f, vv_f, dv_f = sim_vite(start_hand, tgt, G_amp=10, T=1.5)
sv_f = np.linalg.norm(vv_f, axis=1)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))

# A: Hand paths
ax1.plot(pos_m[:,0]*100, pos_m[:,1]*100, '#F39C12', lw=2, label='Min-Jerk (T=500ms)')
ax1.plot(pv_s[:,0]*100, pv_s[:,1]*100, '#2E86AB', lw=2, ls='--', label='VITE G=3')
ax1.plot(pv_f[:,0]*100, pv_f[:,1]*100, '#1B2A4A', lw=2, ls='-.', label='VITE G=10')
ax1.plot(*start_hand*100, 'ko', ms=6); ax1.plot(*tgt*100, 'r*', ms=10)
ax1.set_aspect('equal'); ax1.legend(fontsize=8)
ax1.set_title('A. Hand Paths'); ax1.set_xlabel('x (cm)'); ax1.set_ylabel('y (cm)')

# B: Speed profiles
ax2.plot(tv_s*1000, sv_s*100, '#2E86AB', lw=2, ls='--', label='VITE G=3 (slow)')
ax2.plot(t_m*1000, spd_m*100, '#F39C12', lw=2.5, label='Min-Jerk (T=500ms)')
ax2.plot(tv_f*1000, sv_f*100, '#1B2A4A', lw=2, ls='-.', label='VITE G=10 (fast)')
ax2.set_xlim(0, 1000); ax2.legend(fontsize=8)
ax2.set_title('B. Speed Profiles'); ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Speed (cm/s)')

# C: |DV| = |T - P| (derived from state P, not an independent variable)
ax3.plot(tv_s*1000, dv_s*100, '#2E86AB', lw=2, ls='--', label='|DV| G=3')
ax3.plot(tv_f*1000, dv_f*100, '#1B2A4A', lw=2, ls='-.', label='|DV| G=10')
ax3.set_xlim(0, 1000); ax3.legend(fontsize=8)
ax3.set_title('C. |T − P| (derived from state P)'); ax3.set_xlabel('Time (ms)'); ax3.set_ylabel('|DV| (cm)')

fig.suptitle('Figure 4: VITE vs Min-Jerk — Single Reach (8 cm rightward)', fontweight='bold', y=1.02)
plt.tight_layout()

---
## Part 6: VITE Speed Scaling (Lecture §5 — Figure 5)

One of VITE's most attractive features: changing only G produces the same path at different speeds.

**Your tasks:**
1. Simulate rightward reaches with G = 2, 3, 5, 8, 10
2. Use T = 2.5 s so slow reaches have time to settle
3. Plot hand paths (left panel — they should overlap) and speed profiles (right panel)

In [ ]:
# TODO: COMPLETE THIS CELL
# Reproduce Figure 5 from the lecture notes.
# G_vals = [2, 3, 5, 8, 10]
# Left panel: hand paths (should all overlap)
# Right panel: speed profiles (different peaks, same shape, all → 0)




---
## Part 7: Three-Way Comparison (Lecture §6 — Figure 6)

**Your task:** Create a 3-panel figure showing all three controllers on center-out reaching.
- Left: **EPH** (λ ramps, C = 0.25, same as Part 4)
- Center: **Min-Jerk** (T = 500 ms, same as Part 2)
- Right: **VITE** (G = 10, same `sim_vite` from Part 5)

All 8 directions, same axis limits on all three panels.

In [ ]:
# TODO: COMPLETE THIS CELL
# Reproduce Figure 6 from the lecture notes.
# fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
# Loop over targets_8, run all three controllers, plot in each panel.




---
## Part 8: Via-Point Movements (Lecture §8 — Figure 8)

How do the controllers handle intermediate targets?
- **Min-jerk:** Two concatenated 250 ms polynomial segments
- **VITE:** Target switches from via-point to final target at 350 ms  
- **EPH:** 150 ms λ ramps with a 100 ms dwell (50–200 ms to via, 200–300 ms dwell, 300–450 ms to final). EPH needs the dwell because there is no internal mechanism to decide "close enough" — the programmer must specify the timing.

In [ ]:
via_pt   = start_hand + np.array([0.04, 0.06])
final_pt = start_hand + np.array([0.08, 0.0])

# ── Min-jerk: two segments ──
T_seg = 0.250
t1, pos1, vel1 = minimum_jerk(start_hand, via_pt, T_seg)
t2, pos2, vel2 = minimum_jerk(via_pt, final_pt, T_seg)
mj_pos = np.vstack([pos1, pos2[1:]])
mj_vel = np.vstack([vel1, vel2[1:]])
mj_t   = np.concatenate([t1, t2[1:] + T_seg])
mj_spd = np.linalg.norm(mj_vel, axis=1)

# ── VITE: target switch at 250 ms (when hand approaches via-point) ──
def sim_vite_via(p0, via, final, G_amp, switch_t, T):
    alpha = 4.0 * G_amp
    state = np.array([p0[0], p0[1], 0.0, 0.0])
    n = int(T / DT_VITE)
    ts = np.zeros(n); pos = np.zeros((n, 2)); vel = np.zeros((n, 2))
    for i in range(n):
        t = i * DT_VITE; ts[i] = t
        pos[i] = state[:2]; vel[i] = state[2:]
        target = via if t < switch_t else final
        state = rk4_step(state,
            lambda s, tg=target, _G=G_amp, _a=alpha: vite_deriv(s, tg, _G, _a), DT_VITE)
    return ts, pos, vel

tv_via, pv_via, vv_via = sim_vite_via(start_hand, via_pt, final_pt, 10, 0.25, 1.0)
sv_via = np.linalg.norm(vv_via, axis=1)

# ── EPH: 150 ms ramps with 100 ms dwell at via-point ──
via_q   = ik(via_pt[0], via_pt[1])
final_q = ik(final_pt[0], final_pt[1])
li = lambda_for_posture(Q_REF)
lv = lambda_for_posture(via_q)
lf = lambda_for_posture(final_q)

def eph_via_ramp(t):
    if t < 0.05:   return li.copy()
    elif t < 0.20: return li + (t - 0.05) / 0.15 * (lv - li)
    elif t < 0.30: return lv.copy()       # 100 ms dwell at via
    elif t < 0.45: return lv + (t - 0.30) / 0.15 * (lf - lv)
    else:          return lf.copy()

te_via, _, he_via, _ = simulate_lambda(eph_via_ramp, T=1.0)
spe_via = np.zeros(len(te_via))
spe_via[1:] = np.linalg.norm(np.diff(he_via, axis=0) / 0.0001, axis=1)

# ── Plot ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(mj_pos[:,0]*100, mj_pos[:,1]*100, '#F39C12', lw=2, label='Min-Jerk')
ax1.plot(pv_via[:,0]*100, pv_via[:,1]*100, '#2E86AB', lw=2, ls='--', label='VITE')
ax1.plot(he_via[:,0]*100, he_via[:,1]*100, '#E74C3C', lw=2, ls=':', label='EPH')
ax1.plot(*start_hand*100, 'ko', ms=6)
ax1.plot(via_pt[0]*100, via_pt[1]*100, 's', color='#666', ms=8, label='Via-point')
ax1.plot(final_pt[0]*100, final_pt[1]*100, 'r*', ms=10, label='Target')
ax1.set_aspect('equal'); ax1.legend(fontsize=7, loc='lower right')
ax1.set_title('A. Via-Point Hand Paths', fontweight='bold')
ax1.set_xlabel('x (cm)'); ax1.set_ylabel('y (cm)')

ax2.plot(mj_t*1000, mj_spd*100, '#F39C12', lw=2, label='Min-Jerk')
ax2.plot(tv_via*1000, sv_via*100, '#2E86AB', lw=2, ls='--', label='VITE')
ax2.plot(te_via*1000, spe_via*100, '#E74C3C', lw=2, ls=':', label='EPH')
ax2.set_xlim(0, 800); ax2.legend(fontsize=8)
ax2.set_title('B. Via-Point Velocity', fontweight='bold')
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Speed (cm/s)')

fig.suptitle('Figure 8: Via-Point Reaching — Three Controllers', fontweight='bold', y=1.02)
plt.tight_layout()

---
## Part 9: Velocity Comparison (Lecture §9 — Figure 9)

The final comparison: all four velocity traces on one plot for a single rightward reach.

**Your task:** Create Figure 9 from the lecture notes — one panel with:
- EPH (λ ramp) 
- VITE G = 3 (slow)
- Min-Jerk (T = 500 ms)
- VITE G = 10 (fast)

The min-jerk peak should fall between the two VITE speeds.

In [ ]:
# TODO: COMPLETE THIS CELL
# Reproduce Figure 9 from the lecture notes.
# Single rightward reach, 4 traces on one axis.
# Use the same EPH reach from Part 4 (0° direction) and
# the VITE/min-jerk data from Part 5b (or recompute here).




---
## Summary

1. **Morasso's observation (§1):** Straight hand paths and bell-shaped velocity in task space, but complex joint trajectories — the nervous system appears to plan in task coordinates.

2. **Minimum jerk (§2):** Explains straight paths perfectly — but is pure feedforward with no perturbation response. The purest "Initial Impulse."

3. **EPH vs min-jerk (§3):** EPH produces the right qualitative behavior but the wrong geometry. Min-jerk produces the right geometry but says nothing about muscles.

4. **VITE (§4):** Neural dynamics that produce min-jerk-like trajectories without optimization. DV = T − P is derived from state P, not an independent variable.

5. **Speed scaling (§5):** The Go signal G is a single "vigor" parameter — changes speed without changing the path.

6. **Three-way (§6):** EPH curves, min-jerk straight, VITE near-straight. Simplicity of control trades off against kinematic invariance.

7. **Via-points (§8):** VITE handles via-points most elegantly (target switch). Min-jerk needs re-optimization. EPH needs sequential ramps.

**Next week:** If the brain plans in task space (VITE/min-jerk), it must convert to muscle commands. Week 6 introduces **inverse dynamics.**